# Экспорт `trial.params` в JSON для `--params-json`

Формат — плоский объект, как у завершённого trial transformer-поиска (те же ключи, что в Optuna `FrozenTrial.params`). Его читает **`scripts/run_transformer_fixed_params_all_patients.py`**.

Путь к SQLite совпадает с тем, что даёт `run_optuna_transformer_all_patients.py` (`out-dir / tfr_<id>_transformer.db`). Имя study обычно равно stem файла.


In [4]:
import json
import sys
from pathlib import Path
from typing import cast

from optuna.trial import TrialState

_cwd = Path.cwd().resolve()
_cands = [_cwd, _cwd.parent, *_cwd.parents[:3]]
project_root = next((p for p in _cands if (p / "lib" / "optuna").is_dir()), None)
if project_root is None:
    raise FileNotFoundError(
        "NeuronDeCo root not found (expected lib/optuna). cwd=" + str(_cwd)
    )
sys.path.insert(0, str(project_root))

from lib.optuna import load_study_sqlite

# --- полный путь к PreprocessedData и каталогу run (как --out-dir у Optuna-скрипта) ---
PREPROCESSED_ROOT = Path(
    "/beegfs/home/t.samsonov/notebooks/Pirogov/PreprocessedData"
).resolve()

RUN_SUBDIR = "2026-04-01"

# Список пациентов (как --subjects у run_optuna_transformer_all_patients.py)
PATIENT_IDS = [
    "s02",
    "s03",
    "s04",
    "s05",
    "s06",
    "s07",
    "s09",
    "s10",
    "s11",
    "s12",
    "s13",
    "s15",
]

# Если None — правило «top-5 по F1 → минимум второй цели» (одинаково для всех).
TRIAL_NUMBER = cast(int | None, None)


def select_best_trial_top5_f1_then_min_loss(study):
    complete_trials = [
        t
        for t in study.get_trials(deepcopy=False)
        if t.state == TrialState.COMPLETE and t.values is not None and len(t.values) >= 2
    ]
    if not complete_trials:
        return None
    ranked_by_f1 = sorted(complete_trials, key=lambda t: float(t.values[0]), reverse=True)
    top5 = ranked_by_f1[:5]
    return min(top5, key=lambda t: float(t.values[1]))


run_dir = (PREPROCESSED_ROOT / RUN_SUBDIR).resolve()
if not run_dir.is_dir():
    raise FileNotFoundError(f"Каталог run не найден: {run_dir}")

written = []
skipped = {}

for patient_id in PATIENT_IDS:
    study_db = run_dir / f"tfr_{patient_id}_transformer.db"
    study_db = study_db.resolve()
    study_name = study_db.stem

    if not study_db.is_file():
        skipped[patient_id] = f"missing db: {study_db}"
        print("SKIP", patient_id, skipped[patient_id])
        continue

    study = load_study_sqlite(db_path=study_db, study_name=study_name)

    if TRIAL_NUMBER is not None:
        trial = next(
            (t for t in study.get_trials(deepcopy=False) if int(t.number) == TRIAL_NUMBER),
            None,
        )
        if trial is None:
            skipped[patient_id] = f"no trial #{TRIAL_NUMBER}"
            print("SKIP", patient_id, skipped[patient_id])
            continue
    else:
        trial = select_best_trial_top5_f1_then_min_loss(study)
        if trial is None:
            skipped[patient_id] = "no COMPLETE trial with 2 objectives"
            print("SKIP", patient_id, skipped[patient_id])
            continue

    flat = dict(trial.params)
    out_json = study_db.with_name(f"{study_db.stem}_best_params.json")
    meta_path = out_json.with_suffix(".meta.json")

    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(flat, f, indent=2, ensure_ascii=False)

    meta = {
        "study_db": str(study_db),
        "study_name": study_name,
        "trial_number": int(trial.number),
        "trial_values": list(map(float, trial.values)) if trial.values else None,
        "params_json": str(out_json),
    }
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)

    written.append((patient_id, str(out_json)))
    print("OK", patient_id, "trial", trial.number, "->", out_json.name)

print("\n--- summary ---")
print("written:", len(written))
for pid, pth in written:
    print(" ", pid, pth)
if skipped:
    print("skipped:", len(skipped))
    for pid, msg in skipped.items():
        print(" ", pid, msg)

OK s02 trial 41 -> tfr_s02_transformer_best_params.json
OK s03 trial 98 -> tfr_s03_transformer_best_params.json
OK s04 trial 82 -> tfr_s04_transformer_best_params.json
OK s05 trial 55 -> tfr_s05_transformer_best_params.json
OK s06 trial 51 -> tfr_s06_transformer_best_params.json
OK s07 trial 2 -> tfr_s07_transformer_best_params.json
OK s09 trial 92 -> tfr_s09_transformer_best_params.json
OK s10 trial 40 -> tfr_s10_transformer_best_params.json
OK s11 trial 88 -> tfr_s11_transformer_best_params.json
OK s12 trial 85 -> tfr_s12_transformer_best_params.json
OK s13 trial 60 -> tfr_s13_transformer_best_params.json
OK s15 trial 0 -> tfr_s15_transformer_best_params.json

--- summary ---
written: 12
  s02 /beegfs/home/t.samsonov/notebooks/Pirogov/PreprocessedData/2026-04-01/tfr_s02_transformer_best_params.json
  s03 /beegfs/home/t.samsonov/notebooks/Pirogov/PreprocessedData/2026-04-01/tfr_s03_transformer_best_params.json
  s04 /beegfs/home/t.samsonov/notebooks/Pirogov/PreprocessedData/2026-04-01